# Pipeline V8 modular — BERTimbau + ADTC + Taxonomia + RAG integrado + avaliação manual + XAI

Este notebook substitui a execução monolítica por chamadas ao subpacote `adtc.v8`. A lógica está nos módulos do repositório; aqui permanecem apenas configuração, execução e inspeção.

In [ ]:
from pathlib import Path
import sys

# Quando aberto a partir de notebooks/v8/, adiciona a raiz do projeto ao path.
RAIZ = Path.cwd()
while RAIZ.name != "adtc" and RAIZ.parent != RAIZ:
    RAIZ = RAIZ.parent
sys.path.insert(0, str(RAIZ))

from adtc.v8 import ConfigV8, PathsV8, PipelineV8


## 1. Caminhos e configuração

Use a cópia autorizada do IDPT. O teste deve conter, preferencialmente, as colunas `marcadores` e `n_marc_sup` da etapa anterior da ADTC.

In [ ]:
paths = PathsV8(
    treino=RAIZ / "data/v8/idpt/training_news.csv",
    teste=RAIZ / "data/v8/idpt/idpt_news_teste_adtc_corrigido.csv",
    predicoes_teste=RAIZ / "data/v8/idpt/idpt_news_predicoes_teste.csv",
    taxonomia=RAIZ / "data/v8/taxonomia/quadro_taxonomia_ironia_pb_integrado.csv",
    base_rag_curada=RAIZ / "data/v8/base_rag/base_news_rag_curada_v2_com_proveniencia.csv",
    saida=RAIZ / "outputs/v8",
    # avaliacao_manual=RAIZ / "outputs/v8/amostra_avaliacao_manual_rag_integrado_adtc_v8_preenchida.xlsx",
    # xai_sintese=RAIZ / "outputs/v8/xai_sintese_v8.csv",
)

config = ConfigV8(
    usar_neural=True,
    usar_reranker=True,
    modo_post_hoc=True,   # reprodução da tese; use False em produção
    executar_xai=False,
)


## 2. Execução integral

A chamada abaixo executa: integridade → BERTimbau pré-computado → taxonomia → zonas ADTC → base RAG integrada → recuperação → amostra manual → planilha final.

In [ ]:
resultado = PipelineV8(paths, config).executar()
resultado.arquivos


## 3. Diagnósticos principais

In [ ]:
display(resultado.diagnostico_integridade)
display(resultado.metricas_bert)
display(resultado.matriz_bert)
display(resultado.diagnostico_rag)


## 4. Planilha manual

A primeira execução gera `amostra_avaliacao_manual_rag_integrado_adtc_v8.xlsx`. Preencha-a, informe o caminho em `PathsV8.avaliacao_manual` e reexecute. O pipeline integrará os julgamentos e calculará Precision@k e Hit@k.

In [ ]:
display(resultado.amostra_manual.head())
if not resultado.metricas_manuais.empty:
    display(resultado.metricas_manuais)
if not resultado.problemas_avaliacao.empty:
    display(resultado.problemas_avaliacao)


## 5. Inspeção da trilha auditável

In [ ]:
colunas = [c for c in [
    "id_texto_original", "text", "label", "bert_pred", "bert_conf", "tipo_erro",
    "zona_adtc", "marcadores", "taxonomia_recursos", "tipo_contexto_necessario",
    "fragmento_top1_rag_integrado", "origem_contexto_top1_rag_integrado",
    "score_semantico_top1_rag_integrado", "score_discursivo_top1_rag_integrado",
    "score_integrado_top1_rag_integrado", "manual_decisao_sobre_contexto",
] if c in resultado.dataset.columns]
display(resultado.dataset[colunas].head(10))
